# Ray Simple Job Submission Tutorial

A short guide to running a `checkmaite` capability asynchronously with the lightweight `ray-simple` job backend.

## When to use this backend

Use `ray-simple` for local notebooks, demos, and single-driver workflows where you want one Ray task per submitted capability run.

Important assumptions:

- job handles live only in the current Python process;
- every submission creates a new Ray task;
- completed results are written to the configured analytics store before `job.result()` succeeds.

## Setup

In [ ]:
import tempfile
from pathlib import Path

from checkmaite.core.analytics_store import AnalyticsStore, ParquetBackend
from checkmaite.core.object_detection import DataevalCleaning
from checkmaite.core.object_detection.dataset_loaders import CocoDetectionDataset
from checkmaite.jobs import (
    configure_job_backend,
    get_job,
    list_jobs,
    shutdown_job_backend,
    submit_capability,
)

# Find the repository root whether the notebook is run from docs/tool-usage or elsewhere.
REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())

# Use the tiny COCO fixture included with the repository.
dataset_root = REPO_ROOT / "tests/data_for_tests/coco_dataset"
dataset_ann = dataset_root / "ann_file.json"

dataset = CocoDetectionDataset(
    root=str(dataset_root),
    ann_file=str(dataset_ann),
    dataset_id="coco-job-tutorial",
)

store_dir = Path(tempfile.mkdtemp(prefix="checkmaite_jobs_")) / "analytics_store"
analytics_store_config = {"backend": "parquet", "uri": str(store_dir)}

print(f"Dataset: {dataset.metadata['id']} ({len(dataset)} images)")
print(f"Analytics store: {store_dir}")

## Configure `ray-simple`

The analytics-store config is forwarded to the worker so the worker knows where to write completed run data.

In [ ]:
configure_job_backend(
    "ray-simple",
    address="local",
    force_reinit=True,
    analytics_store=analytics_store_config,
)

print("Configured ray-simple job backend.")

## Submit a capability job

In [ ]:
capability = DataevalCleaning()

job = submit_capability(
    capability,
    datasets=[dataset],
    use_cache=False,
)

print(f"Job ID: {job.job_id}")
print(f"Initial status: {job.status}")

## Wait for completion and get the result reference

`job.result()` returns a `CapabilityRunRef`, not the full capability output object.

In [ ]:
final_status = job.wait(timeout=120)
print(f"Final status: {final_status}")

ref = job.result(timeout=10)
print(f"Run UID: {ref.run_uid}")
print(f"Store URI: {ref.store_uri}")

## List jobs remembered by this backend object

In [ ]:
for remembered_job in list_jobs(limit=10):
    print(remembered_job.job_id, remembered_job.status)

print("Fetched by ID:", get_job(job.job_id).status)

## Query the analytics store

In [ ]:
store = AnalyticsStore(ParquetBackend(str(store_dir)))

print(f"Tables: {store.list_tables()}")

cleaning_results = store.query_sql("""
    SELECT
        dataset_id,
        exact_duplicate_count,
        image_outlier_count,
        image_outlier_ratio,
        mean_brightness
    FROM dataeval_cleaning
""")
print(cleaning_results)

## Shut down

In [ ]:
shutdown_job_backend(wait=True)
print("Job backend shut down.")